In [9]:
import pandas as pd

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from string import punctuation
news = pd.read_csv("../../data/validation/news.tsv", sep="\t", names=['news_id', 'category', 'subcategory', 'title', 'abstract', 'url', 'title_entities', 'abstract_entities'])
news = news[['news_id', 'category', 'subcategory', 'title', 'abstract']]
news.fillna('', inplace=True) 


news['text'] = news['category'] + ' ' + news['subcategory'] + ' ' + news['title'] + ' ' + news['abstract']
news = news[['news_id', 'text']]

stop_words = set(stopwords.words('english'))
stop_words.update(punctuation)
ps = PorterStemmer()
def clean_text(text):
    words = word_tokenize(text)
    words = [ps.stem(w) for w in words if w not in stop_words]
    return ' '.join(words)

news['text'] = news['text'].apply(clean_text)


news.head()


,news_id,text
0,N55528,lifestyl lifestyleroy the brand queen elizabeth princ charl princ philip swear by shop notebook jacket royal ca n't live without
1,N18955,health medic dispos unwant prescript drug dea 's take back day
2,N61837,news newsworld the cost trump 's aid freez trench ukrain 's war lt. ivan molchanet peek parapet sand bag front line war ukrain next empti helmet prop trick sniper alreadi perfor multipl hole
3,N53526,health voic i wa an nba wife here 's how it affect my mental health i felt like i fraud nba wife n't help in fact nearli destroy
4,N38324,health medic how get rid skin tag accord dermatologist they seem harmless 's good reason n't ignor the post how get rid skin tag accord dermatologist appear first reader 's digest


In [14]:
behaviors = pd.read_csv('../../data/validation/behaviors.tsv', sep='\t', names=['impression_id', 'user_id', 'time', 'history', 'impressions'])
behaviors.fillna('', inplace=True)
behaviors['history'] = behaviors['history'].apply(lambda x: x.split(' '))
behaviors['impressions'] = behaviors['impressions'].apply(lambda x: x.split(' '))
behaviors['impressions'] = behaviors['impressions'].apply(lambda x: [i.split('-') for i in x])
behaviors.head()



,impression_id,user_id,time,history,impressions
0,1,U80234,11/15/2019 12:37:50 PM,"[N55189, N46039, N51741, N53234, N11276, N264, N40716, N28088, N43955, N6616, N47686, N63573, N38895, N30924, N35671]","[[N28682, 0], [N48740, 0], [N31958, 1], [N34130, 0], [N6916, 0], [N5472, 0], [N50775, 0], [N24802, 0], [N19990, 0], [N33176, 0], [N62365, 0], [N5940, 0], [N6400, 0], [N58098, 0], [N42844, 0], [N49285, 0], [N51470, 0], [N53572, 0], [N11930, 0], [N21679, 0], [N55237, 0], [N29862, 0]]"
1,2,U60458,11/15/2019 7:11:50 AM,"[N58715, N32109, N51180, N33438, N54827, N28488, N61186, N34775, N33742, N50020, N57061, N30924, N6778]","[[N20036, 0], [N23513, 1], [N32536, 0], [N46976, 0], [N35216, 0], [N36779, 0], [N31958, 0]]"
2,3,U44190,11/15/2019 9:55:12 AM,"[N56253, N1150, N55189, N16233, N61704, N51706, N53033, N15634, N3259]","[[N36779, 0], [N62365, 0], [N58098, 0], [N5472, 0], [N13408, 0], [N55036, 0], [N19990, 0], [N53283, 0], [N20036, 0], [N47383, 0], [N37352, 0], [N31958, 0], [N50775, 0], [N5940, 1], [N58251, 0], [N49285, 0], [N30290, 0], [N11930, 0], [N16680, 0], [N42844, 0], [N53572, 0], [N6916, 0], [N55237, 0]]"
3,4,U87380,11/15/2019 3:12:46 PM,"[N63554, N49153, N28678, N23232, N43369, N58518, N44402, N7649, N63429, N45794, N53531, N53033, N30765, N34452, N24298, N29361, N2597, N28926, N28247]","[[N6950, 0], [N60215, 0], [N6074, 0], [N11930, 0], [N6916, 0], [N24802, 0], [N48740, 0], [N60675, 0], [N45057, 0], [N51470, 0], [N62365, 0], [N15347, 1], [N21941, 0], [N49285, 0], [N29091, 0], [N6400, 0], [N19611, 0], [N52492, 0], [N29862, 0], [N19990, 0], [N38620, 0], [N60762, 0], [N34130, 0], [N1952, 0], [N53572, 0], [N4733, 0]]"
4,5,U9444,11/15/2019 8:25:46 AM,"[N51692, N18285, N26015, N22679, N55556]","[[N5940, 1], [N23513, 0], [N49285, 0], [N23355, 0], [N19990, 0], [N31958, 1], [N29393, 0], [N30290, 0], [N19611, 0], [N62365, 0], [N51470, 0], [N34130, 0], [N36779, 0], [N20036, 0]]"


In [15]:
# group behaviors by user_id and add two columns: pos and neg where pos is all news in history and all impressions that contain -1 and 
# neg is all impressions that do not contain -1
pd.set_option('display.max_colwidth', None)
behaviors = behaviors.groupby('user_id').agg({'history': 'sum', 'impressions': 'sum'}).reset_index()
behaviors['history'] = behaviors['history'].apply(lambda x: list(set(x)))
behaviors['neg'] = behaviors['impressions'].apply(lambda x: [i[0] for i in x if i[1] == '0'])
behaviors['pos'] = behaviors['impressions'].apply(lambda x: [i[0] for i in x if i[1] == '1'])
behaviors['pos'] = behaviors['pos'] + behaviors['history']
behaviors['pos'] = behaviors['pos'].apply(lambda x: list(set(x)))
behaviors['neg'] = behaviors['neg'].apply(lambda x: list(set(x)))
behaviors = behaviors[['user_id', 'pos', 'neg']]



In [16]:
pos = behaviors[['user_id', 'pos']].explode('pos').rename(columns={'pos': 'news_id'})

neg = behaviors[['user_id', 'neg']].explode('neg').rename(columns={'neg': 'news_id'})
pos['label'] = 1
neg['label'] = 0

behaviors = pd.concat([pos, neg])
behaviors = behaviors.merge(news, on='news_id', how='left')
behaviors = behaviors.sample(frac=1, random_state=42).reset_index(drop=True)



print(behaviors)

        user_id news_id  label  \
0        U32158  N58656      0   
1         U4379  N36786      0   
2        U45248  N48487      0   
3        U80965  N61197      0   
4        U29518  N21679      0   
...         ...     ...    ...   
3739361  U44988  N12446      0   
3739362  U86385  N59904      0   
3739363  U40754   N4390      0   
3739364  U59829  N58612      0   
3739365  U40461  N13270      0   

                                                                                                                                                                                                                                                                                  text  
0                                                                                                                                                                           movi movies-galleri 'charli 's angel star where we 're check former charli 's angel actress find 're today  
1                            

In [17]:
behaviors = behaviors[['user_id', 'text', 'label', 'news_id']]

behaviors.to_csv('test.csv', index=False)